# 01 - Setup, data preparation, baseline generation, SFT
Accelerator: **GPU T4 x2**, Internet **on**.

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Private repo: add a GitHub token as a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets -> Add secret, then tick it for this notebook).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import os, subprocess, sys
if not os.path.exists(REPO):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # don't keep the token on disk
    del token, authed
os.chdir(REPO)
!git -C {REPO} log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

## Public + own adapter tests

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/

## Data (UltraChat SFT, HH-RLHF, UltraFeedback, AlpacaEval 300 / seed 42)

In [ ]:
!python scripts/prepare_data.py
!wc -l data/processed/*.jsonl
!ls -l eval_sets/

## Task 2: baseline generation (Qwen2.5-0.5B base) on GPU 1 while SFT trains on GPU 0

In [ ]:
import subprocess
base_log = open("/kaggle/working/base_gen.log", "w")
base_proc = subprocess.Popen(
    "python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --device cuda:1 "
    "--eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-base "
    "--dataset_name alpaca_eval --output_path outputs/generations/alpaca/base.json && "
    "python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --device cuda:1 "
    "--eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-base "
    "--output_path outputs/generations/hh/base.json",
    shell=True, stdout=base_log, stderr=subprocess.STDOUT)

## Task 3: SFT (30k train / 1k val)

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft --device cuda:0

In [ ]:
base_proc.wait(); print("base generation exit code", base_proc.returncode)
!tail -3 /kaggle/working/base_gen.log
!python scripts/plot_metrics.py --sft outputs/sft/metrics.jsonl --out_dir report/figures

## Sanity check the SFT checkpoint on a few prompts

In [ ]:
!python scripts/generate.py --model_path outputs/sft/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name sft-sanity --output_path /tmp/x.json --limit 4

In [ ]:
# Archive outputs so they survive the session (then: Save Version -> attach as dataset).
!cd {REPO} && zip -qr /kaggle/working/sft_run_outputs.zip outputs/sft data/processed eval_sets outputs/generations report/figures
!ls -lh /kaggle/working/*.zip